<a href="https://colab.research.google.com/github/quain7/stm32-project/blob/quain7-patch-3/RQT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import numpy as np
import pandas as pd
import scipy.signal
import scipy.interpolate
import gpxpy
import gpxpy.geo
import pywt
import folium
from hmmlearn import hmm
from ahrs.filters import Madgwick
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler

def q2R(q):
    q0, q1, q2, q3 = q
    return np.array([
        [1 - 2*q2**2 - 2*q3**2, 2*q1*q2 - 2*q0*q3, 2*q1*q3 + 2*q0*q2],
        [2*q1*q2 + 2*q0*q3, 1 - 2*q1**2 - 2*q3**2, 2*q2*q3 - 2*q0*q1],
        [2*q1*q3 - 2*q0*q2, 2*q2*q3 + 2*q0*q1, 1 - 2*q1**2 - 2*q2**2]
    ])

def main():

    # 1. Data Loading
    print("Loading datasets...")

    imu_columns = ['timestamp_ms', 'ax', 'ay', 'az', 'gx', 'gy', 'gz']
    try:
        df_imu = pd.read_csv('ROUTE009.CSV', names=imu_columns)

        df_imu = df_imu.apply(pd.to_numeric, errors='coerce').dropna().reset_index(drop=True)

    except FileNotFoundError:
        print("Warning: TEST (2).TXT not found. Please ensure the file exists.")
        return

    # Convert LSB to physical units (±2G & ±250dps)
    df_imu[['ax', 'ay', 'az']] = df_imu[['ax', 'ay', 'az']] / 16384.0
    dps_to_rads = np.pi / 180.0
    df_imu[['gx', 'gy', 'gz']] = (df_imu[['gx', 'gy', 'gz']] / 131.0) * dps_to_rads

    try:
        with open('20260707-180605.gpx', 'r') as gpx_file:
            gpx = gpxpy.parse(gpx_file)

        gps_data = []
        for track in gpx.tracks:
            for segment in track.segments:
                for point in segment.points:
                    gps_data.append({
                        'time': point.time,
                        'lat': point.latitude,
                        'lon': point.longitude,
                        'ele': point.elevation
                    })
        df_gps = pd.DataFrame(gps_data)
        df_gps['time'] = pd.to_datetime(df_gps['time'], utc=True)
    except FileNotFoundError:
        print("Warning: route.gpx not found. Please ensure the file exists.")
        return

    # 2. Time Synchronization

    print("Synchronizing timeframes...")

    accel_mag = np.sqrt(df_imu['ax']**2 + df_imu['ay']**2 + df_imu['az']**2)
    rolling_var = accel_mag.rolling(window=100, min_periods=10).var()

    noise_threshold = 0.005
    movement_mask = rolling_var > noise_threshold

    if not movement_mask.any():
        raise ValueError("No movement detected in IMU data to establish anchor.")

    imu_start_idx = movement_mask.idxmax()
    imu_anchor_ms = df_imu.loc[imu_start_idx, 'timestamp_ms']

    # --- GPS Anchor ---
    distances = [0.0]
    for i in range(1, len(df_gps)):
        p1 = gpxpy.geo.Location(df_gps.loc[i-1, 'lat'], df_gps.loc[i-1, 'lon'])
        p2 = gpxpy.geo.Location(df_gps.loc[i, 'lat'], df_gps.loc[i, 'lon'])
        distances.append(p1.distance_2d(p2))

    df_gps['dist'] = distances
    df_gps['dt'] = df_gps['time'].diff().dt.total_seconds().fillna(0)
    df_gps['speed'] = np.where(df_gps['dt'] > 0, df_gps['dist'] / df_gps['dt'], 0)

    speed_threshold = 1.0 # m/s
    gps_movement_mask = df_gps['speed'] > speed_threshold
    if not gps_movement_mask.any():
        raise ValueError("No movement detected in GPS data to establish anchor.")

    gps_start_idx = gps_movement_mask.idxmax()
    gps_anchor_time = df_gps.loc[gps_start_idx, 'time']

    # --- Alignment & Interpolation ---
    base_timestamp = gps_anchor_time - pd.to_timedelta(imu_anchor_ms, unit='ms')
    df_imu['abs_time'] = base_timestamp + pd.to_timedelta(df_imu['timestamp_ms'], unit='ms')

    df_gps['time_sec'] = df_gps['time'].astype(np.int64) / 1e9
    df_imu['time_sec'] = df_imu['abs_time'].astype(np.int64) / 1e9

    df_gps_clean = df_gps.drop_duplicates(subset=['time_sec']).sort_values('time_sec')

    interp_lat = scipy.interpolate.interp1d(
        df_gps_clean['time_sec'], df_gps_clean['lat'],
        kind='linear', bounds_error=False,
        fill_value=(df_gps_clean['lat'].iloc[0], df_gps_clean['lat'].iloc[-1])
    )
    interp_lon = scipy.interpolate.interp1d(
        df_gps_clean['time_sec'], df_gps_clean['lon'],
        kind='linear', bounds_error=False,
        fill_value=(df_gps_clean['lon'].iloc[0], df_gps_clean['lon'].iloc[-1])
    )

    df_imu['lat'] = interp_lat(df_imu['time_sec'])
    df_imu['lon'] = interp_lon(df_imu['time_sec'])


    # 3. Calibration

    print("Calibrating and running Madgwick AHRS...")

    stat_mask = df_imu['timestamp_ms'] < imu_anchor_ms

    gx_bias = df_imu.loc[stat_mask, 'gx'].mean()
    gy_bias = df_imu.loc[stat_mask, 'gy'].mean()
    gz_bias = df_imu.loc[stat_mask, 'gz'].mean()

    ax_bias = df_imu.loc[stat_mask, 'ax'].mean()
    ay_bias = df_imu.loc[stat_mask, 'ay'].mean()
    az_bias = df_imu.loc[stat_mask, 'az'].mean() - 1.0

    df_imu['gx'] -= gx_bias
    df_imu['gy'] -= gy_bias
    df_imu['gz'] -= gz_bias
    df_imu['ax'] -= ax_bias
    df_imu['ay'] -= ay_bias
    df_imu['az'] -= az_bias

    dt_ahrs = 1.0 / 100.0 # 100 Hz


    madgwick = Madgwick(frequency=100.0)

    Q = np.zeros((len(df_imu), 4))
    Q[0] = [1.0, 0.0, 0.0, 0.0]

    gyr_data = df_imu[['gx', 'gy', 'gz']].values
    acc_data = df_imu[['ax', 'ay', 'az']].values

    for i in range(1, len(df_imu)):

        Q[i] = madgwick.updateIMU(Q[i-1], gyr_data[i], acc_data[i])

    a_global = np.zeros((len(df_imu), 3))
    for i in range(len(df_imu)):
        R = q2R(Q[i])
        a_global[i] = R @ acc_data[i]

    dyn_az = a_global[:, 2] - 1.0
    df_imu['dyn_az'] = dyn_az


    # 4 DSP and HMM Classification


    print("Applying High-Pass filter to strictly remove gravity leakage...")
    sos_hp = scipy.signal.butter(4, 0.5, 'hp', fs=100, output='sos')
    az_filtered = scipy.signal.sosfilt(sos_hp, df_imu['az'].fillna(0))

    print("Extracting frequency bands (Welch's PSD) and Kurtosis...")
    window_size = 100
    features = []
    chunk_indices = []

    for i in range(0, len(az_filtered) - window_size, window_size):
        window = az_filtered[i:i+window_size]

        kurt = kurtosis(window, fisher=True)
        if np.isnan(kurt): kurt = 0

        freqs, psd = scipy.signal.welch(window, fs=100, nperseg=window_size)

        band_low = np.sum(psd[(freqs >= 0.5) & (freqs < 3.0)])
        band_mid = np.sum(psd[(freqs >= 3.0) & (freqs < 10.0)])
        band_high = np.sum(psd[(freqs >= 10.0) & (freqs <= 40.0)])

        total_e = band_low + band_mid + band_high

        features.append([band_low, band_mid, band_high, kurt, total_e])
        chunk_indices.append(i)

    features = np.array(features)

    df_windows = pd.DataFrame({
        'lat': df_imu['lat'].iloc[chunk_indices].values,
        'lon': df_imu['lon'].iloc[chunk_indices].values,
        'time_sec': df_imu['time_sec'].iloc[chunk_indices].values
    })

    print("Calculating GPS speed in km/h...")
    speeds = [0.0]
    for i in range(1, len(df_windows)):
        pt1 = gpxpy.geo.Location(df_windows['lat'].iloc[i-1], df_windows['lon'].iloc[i-1])
        pt2 = gpxpy.geo.Location(df_windows['lat'].iloc[i], df_windows['lon'].iloc[i])
        dist = pt1.distance_2d(pt2)
        dt = df_windows['time_sec'].iloc[i] - df_windows['time_sec'].iloc[i-1]
        speeds.append((dist / dt) * 3.6 if dt > 0 else 0.0)

    df_windows['speed_kmh'] = speeds

    print("Filtering out handling noise...")
    # 1. Відрізаємо час, коли GPS вже не записував
    valid_time = (df_windows['time_sec'] >= df_gps_clean['time_sec'].min()) & \
                 (df_windows['time_sec'] <= df_gps_clean['time_sec'].max())


    valid_speed = df_windows['speed_kmh'] >= 1.0

    valid_mask = valid_time & valid_speed
    df_windows = df_windows[valid_mask].copy().reset_index(drop=True)
    features = features[valid_mask]

    X_train = features[:, :4]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    print("Training Gaussian HMM on advanced spectral features...")
    model = hmm.GaussianHMM(n_components=5, covariance_type="diag", n_iter=100, random_state=42)
    model.fit(X_scaled)
    hidden_states = model.predict(X_scaled)

    print("Logically identifying road states based on total cluster energy...")
    cluster_energies = []
    for state in range(5):
        mask = (hidden_states == state)
        if np.sum(mask) > 0:
            mean_energy = np.mean(features[mask, 4])
        else:
            mean_energy = 0
        cluster_energies.append(mean_energy)

    sorted_states = np.argsort(cluster_energies)

    labels_map = {
        sorted_states[0]: 'Smooth',
        sorted_states[1]: 'Worn',
        sorted_states[2]: 'Cobblestone',
        sorted_states[3]: 'Speedbump',
        sorted_states[4]: 'Pothole'
    }

    df_windows['label'] = [labels_map[state] for state in hidden_states]

    print("\n--- NEW DSP PIPELINE SUMMARY ---")
    print(df_windows['label'].value_counts())
    print("--------------------------------\n")

    # 5. Geospatial Visualization

    print("Generating Folium Map visualization...")

    start_lat = df_windows['lat'].iloc[0]
    start_lon = df_windows['lon'].iloc[0]
    m = folium.Map(location=[start_lat, start_lon], zoom_start=18)

    color_map = {
        'Smooth': '#00FF00',
        'Worn': '#ADFF2F',
        'Cobblestone': '#FFA500',
        'Speedbump': '#FF0000',
        'Pothole': '#8B0000'
    }

    for i in range(len(df_windows) - 1):
        loc1 = [df_windows['lat'].iloc[i], df_windows['lon'].iloc[i]]
        loc2 = [df_windows['lat'].iloc[i+1], df_windows['lon'].iloc[i+1]]
        label = df_windows['label'].iloc[i]

        folium.PolyLine(
            [loc1, loc2],
            color=color_map[label],
            weight=6,
            opacity=0.8,
            popup=label
        ).add_to(m)

    m.save('road_quality_map.html')
    print("Pipeline complete! Visualization saved to 'road_quality_map.html'.")

if __name__ == "__main__":
    main()

Loading datasets...
Synchronizing timeframes...
Calibrating and running Madgwick AHRS...
Applying High-Pass filter to strictly remove gravity leakage...
Extracting frequency bands (Welch's PSD) and Kurtosis...
Calculating GPS speed in km/h...
Filtering out handling noise...
Training Gaussian HMM on advanced spectral features...
Logically identifying road states based on total cluster energy...

--- NEW DSP PIPELINE SUMMARY ---
label
Worn           259
Smooth         242
Cobblestone    130
Pothole          1
Speedbump        1
Name: count, dtype: int64
--------------------------------

Generating Folium Map visualization...
Pipeline complete! Visualization saved to 'road_quality_map.html'.


In [3]:
pip install numpy pandas scipy gpxpy PyWavelets folium hmmlearn ahrs requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.7/244.7 kB 8.9 MB/s eta 0:00:00
